## Ch10. Dynamic regression models Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-10-dynamic-regression.qmd

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## [Slide 5] StatsForecast Implementation

In [2]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA

Automatic ARIMA order selection for errors

In [3]:
us_change = pd.read_csv("data/US_change.csv", parse_dates=["ds"]).rename(columns={"y": "Consumption"})
y_df = us_change[["unique_id", "ds", "Consumption"]]
x_df = us_change[["unique_id", "ds", "Income"]]


In [4]:
sf.fit(df=y_df.merge(x_df, on=["unique_id", "ds"]).rename(columns={"Consumption": "y"}))


NameError: name 'sf' is not defined

## [Slide 8] US Consumption: Code

In [5]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.utils import AirPassengersDF

us_change = pd.read_csv("data/US_change.csv", parse_dates=["ds"])
y_df = us_change[["unique_id", "ds", "Consumption"]]
x_df = us_change[["unique_id", "ds", "Income"]]

sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq='QS'
)
sf.fit(df=y_df.rename(columns={"Consumption": "y"}),
       X_df=x_df.rename(columns={"Income": "x"}))

summary = sf.fitted_[0, 0].summary()
print(summary)

KeyError: "['Consumption'] not in index"

## [Slide 12] Forecasting: US Consumption

In [6]:
from utilsforecast.processing import make_future_dataframe

Assumption: future income changes = historical mean

In [7]:
uids = us_change["unique_id"].unique()
last_times = us_change.groupby("unique_id")["ds"].max()


In [8]:
future_predictors = (
    make_future_dataframe(uids, last_times, freq="QS", h=8)
    .assign(Income=us_change["Income"].mean())
)

fc = sf.forecast(
    df=us_change, h=8,
    X_df=future_predictors,
    target_col="Consumption",
    level=[80, 95]
)

NameError: name 'sf' is not defined

## [Slide 14] Forecasting: Electricity Demand 14-day ahead forecast with constant 26°C temperature

In [9]:
future_elec = pd.DataFrame({
    "unique_id": "VIC",
    "ds": pd.date_range("2015-01-01", periods=14, freq="D"),
    "Temperature": 26.0,
    "Temperature_sq": 26.0**2,
    "Weekday": [1,1,1,1,1,0,0,  # Mon-Sun
                1,1,1,1,1,0,0],
})

fc_elec = sf.forecast(
    df=elec_df, h=14,
    X_df=future_elec,
    level=[80, 95]
)

NameError: name 'sf' is not defined

## [Slide 20] Example: Australian Café/Restaurant Expenditure

In [10]:
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial

Test K = 1, 2, ..., 6

In [11]:
aus_cafe = (
    pd.read_csv("data/aus_retail.csv", parse_dates=["Month"])
    .loc[lambda x: x["Industry"].isin([
        "Cafes, restaurants and catering services",
        "Cafes, restaurants and takeaway food services",
    ])]
    .groupby("Month", as_index=False)["Turnover"].sum()
    .rename(columns={"Month": "ds", "Turnover": "y"})
    .assign(unique_id="aus_cafe")
)


In [12]:
for k in range(1, 7):
    features = [partial(fourier, season_length=12, k=k)]
    aus_cafe_ft, fut_ft = pipeline(
        aus_cafe, features=features, freq="MS", h=24
    )
    sf = StatsForecast(
        models=[AutoARIMA(season_length=1, d=0)],
        freq='MS'
    )
    sf.fit(df=aus_cafe_ft)
    print(f"K={k}: AICc = {sf.fitted_[0,0].model_['ic']:.2f}")

TypeError: unsupported format string passed to NoneType.__format__

## [Slide 26] Insurance Example: Code Create lagged predictor columns

In [13]:
insurance = pd.read_csv("data/insurance.csv", parse_dates=["ds"])


In [14]:
sf.fit(
    df=fit_df[["unique_id","ds","Quotes","TVadverts","TVadverts_1"]].rename(
        columns={"Quotes": "y"})
)


NameError: name 'fit_df' is not defined

Forecast 12 months with advertising = 8 units

In [15]:
future_adv = pd.DataFrame({
    "unique_id": "insurance",
    "ds": pd.date_range("2005-05-01", periods=12, freq="MS"),
    "TVadverts": 8.0,
    "TVadverts_1": [insurance["TVadverts"].iloc[-1]] + [8.0]*11,
})
fc = sf.forecast(h=12, X_df=future_adv, level=[80, 95])

TypeError: StatsForecast.forecast() missing 1 required positional argument: 'df'